In [43]:
import paddle
from paddleocr import PaddleOCR
import fitz
import os
import cv2
import numpy as np
from PIL import Image

In [44]:
ocr = PaddleOCR(use_angle_cls=True, lang='en', det_db_unclip_ratio=2.0)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_23072\3794847802.py:1: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  ocr = PaddleOCR(use_angle_cls=True, lang='en', det_db_unclip_ratio=2.0)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_23072\3794847802.py:1: DeprecationWarning: The parameter `det_db_unclip_ratio` has been deprecated and will be removed in the future. Please use `text_det_unclip_ratio` instead.
  ocr = PaddleOCR(use_angle_cls=True, lang='en', det_db_unclip_ratio=2.0)
Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Using official model (PP-LCNet_x1_0_doc_ori), the model files will be automatically downloaded and saved in `C:\Users\ADMIN\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Using official model (PP-LCNet_x1_0_doc_ori), the model files will be automatically downloaded and saved in `C:\Users\ADMIN\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Fetching 6 files:

In [45]:
def process_file(file_path):
    print(f"\n--- Đang xử lý file: {file_path} ---")
    
    if not os.path.exists(file_path):
        print("Không tìm thấy file.")
        return

    ext = os.path.splitext(file_path)[1].lower()
    full_text = ""

    # === XỬ LÝ PDF ===
    if ext == '.pdf':
        try:
            doc = fitz.open(file_path)
            total_pages = len(doc)
            print(f"Tổng số trang: {total_pages}")

            for i, page in enumerate(doc):
                print(f"  > Đang đọc trang {i+1}/{total_pages}...")
                
                # Zoom=2 (DPI ~144) là đủ cho CPU đọc tốt, tăng lên 3 nếu chữ quá nhỏ
                # Để CPU chạy nhanh hơn, đừng set Zoom quá cao (như 4 hay 5)
                pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
                
                # Convert Pixmap -> Numpy Array
                img_data = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
                if pix.n == 3: 
                    img_data = cv2.cvtColor(img_data, cv2.COLOR_RGB2BGR)
                elif pix.n == 4:
                    img_data = cv2.cvtColor(img_data, cv2.COLOR_RGBA2BGR)
                
                # Gọi OCR
                result = ocr.predict(img_data)
                
                # Lấy text
                page_content = ""
                if result[0]:
                    for line in result[0]:
                        text = line[1][0]
                        page_content += text + " "
                
                full_text += f"--- Trang {i+1} ---{page_content}"
                
            doc.close()
        except Exception as e:
            print(f"Lỗi đọc PDF: {e}")

    # === XỬ LÝ ẢNH ===
    elif ext in ['.jpg', '.jpeg', '.png', '.bmp']:
        result = ocr.predict(file_path)
        if result[0]:
            for line in result[0]:
                text = line[1][0]
                full_text += text + " "
    
    return full_text

In [46]:
input_file = "Screenshot 2025-12-03 221126.png" 
    
if os.path.exists(input_file):
    ket_qua = process_file(input_file)
    print("\n=== KẾT QUẢ ===")
    print(ket_qua)


--- Đang xử lý file: Screenshot 2025-12-03 221126.png ---

=== KẾT QUẢ ===
n a o t o e e e e e e e i e e 

=== KẾT QUẢ ===
n a o t o e e e e e e e i e e 


In [47]:
input_file = "signal.pdf" 
    
if os.path.exists(input_file):
    ket_qua = process_file(input_file)
    print("\n=== KẾT QUẢ ===")
    print(ket_qua)


--- Đang xử lý file: signal.pdf ---
Tổng số trang: 1
  > Đang đọc trang 1/1...

=== KẾT QUẢ ===
--- Trang 1 ---n a o t o e e e e e e e i e e 

=== KẾT QUẢ ===
--- Trang 1 ---n a o t o e e e e e e e i e e 
